# Chunk `contexts_train/dev/test` thành schema tối giản

**Input:** `contexts_train.jsonl`, `contexts_dev.jsonl`, `contexts_test.jsonl`

**Output:** `chunks_train.jsonl`, `chunks_dev.jsonl`, `chunks_test.jsonl`

Mỗi output record chỉ có:

```json
{
  "id": "news_001",
  "context_id": "news_001_chunk_000",
  "content": "Nội dung chunk...",
  "domain": "news"
}
```

Các trường `content_hash`, `context_group_id`, `split` và metadata khác bị bỏ. Tên file thể hiện split.

In [ ]:
from __future__ import annotations

import json
import re
import unicodedata
from collections import Counter
from pathlib import Path
from typing import Any

import pandas as pd
from tqdm.auto import tqdm

INPUT_DIR = Path("D:\\KLTN\\code\\notebooks\\six_domain_binary_output\\03_splits")
OUTPUT_DIR = Path("D:\\KLTN\\code\\notebooks\\six_domain_binary_output\\05_chunks")

INPUT_FILES = {
    "train": INPUT_DIR / "contexts_train.jsonl",
    "dev": INPUT_DIR / "contexts_dev.jsonl",
    "test": INPUT_DIR / "contexts_test.jsonl",
}

CHUNK_MAX_WORDS = 400
CHUNK_OVERLAP_WORDS = 60
MIN_FINAL_CHUNK_WORDS = 100
CHECK_ID_LEAKAGE = True

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

assert 0 <= CHUNK_OVERLAP_WORDS < CHUNK_MAX_WORDS
assert 0 <= MIN_FINAL_CHUNK_WORDS <= CHUNK_MAX_WORDS

In [ ]:
ZERO_WIDTH_RE = re.compile(r"[\u200B-\u200D\uFEFF]")
MULTISPACE_RE = re.compile(r"[ \t]+")
EXCESS_NEWLINES_RE = re.compile(r"\n{3,}")
WORD_RE = re.compile(r"\b\w+\b", flags=re.UNICODE)
SENTENCE_BOUNDARY_RE = re.compile(r"(?<=[.!?…])\s+")


def read_json_or_jsonl(path: Path) -> list[dict[str, Any]]:
    if not path.exists():
        raise FileNotFoundError(f"Không tìm thấy: {path.resolve()}")

    text = path.read_text(encoding="utf-8-sig")
    if not text.strip():
        return []

    try:
        data = json.loads(text)
        records = data if isinstance(data, list) else [data]
    except json.JSONDecodeError:
        records = [
            json.loads(line)
            for line in text.splitlines()
            if line.strip()
        ]

    if not all(isinstance(record, dict) for record in records):
        raise TypeError(f"{path.name} chứa record không phải JSON object.")

    return records


def write_jsonl(records: list[dict[str, Any]], path: Path) -> None:
    with path.open("w", encoding="utf-8") as file:
        for record in records:
            file.write(json.dumps(record, ensure_ascii=False) + "\n")


def normalize_content(text: str) -> str:
    text = unicodedata.normalize("NFC", text)
    text = ZERO_WIDTH_RE.sub("", text)
    text = text.replace("\r\n", "\n").replace("\r", "\n")
    lines = [
        MULTISPACE_RE.sub(" ", line).strip()
        for line in text.split("\n")
    ]
    return EXCESS_NEWLINES_RE.sub("\n\n", "\n".join(lines)).strip()


def count_words(text: str) -> int:
    return len(WORD_RE.findall(text))

## Chunking

Ưu tiên paragraph → sentence. Chỉ cắt theo từ khi một câu dài hơn giới hạn.

In [ ]:
def split_long_unit(text: str, max_words: int) -> list[str]:
    words = text.split()
    if len(words) <= max_words:
        return [text.strip()] if text.strip() else []

    return [
        " ".join(words[start:start + max_words])
        for start in range(0, len(words), max_words)
    ]


def split_into_units(text: str, max_words: int) -> list[str]:
    units: list[str] = []

    paragraphs = [
        paragraph.strip()
        for paragraph in re.split(r"\n\s*\n|\n", text)
        if paragraph.strip()
    ]

    for paragraph in paragraphs:
        sentences = [
            sentence.strip()
            for sentence in SENTENCE_BOUNDARY_RE.split(paragraph)
            if sentence.strip()
        ] or [paragraph]

        for sentence in sentences:
            units.extend(split_long_unit(sentence, max_words))

    return units


def words_in_units(units: list[str]) -> int:
    return sum(count_words(unit) for unit in units)


def pack_chunks(units: list[str], max_words: int) -> list[list[str]]:
    chunks: list[list[str]] = []
    current: list[str] = []
    current_words = 0

    for unit in units:
        unit_words = count_words(unit)

        if current and current_words + unit_words > max_words:
            chunks.append(current)
            current = []
            current_words = 0

        current.append(unit)
        current_words += unit_words

    if current:
        chunks.append(current)

    return chunks


def fix_short_final_chunk(
    chunks: list[list[str]],
    min_words: int,
    max_words: int,
) -> list[list[str]]:
    if len(chunks) <= 1 or min_words <= 0:
        return chunks

    previous, final = chunks[-2], chunks[-1]

    while words_in_units(final) < min_words and len(previous) > 1:
        candidate = previous[-1]
        if words_in_units(final) + count_words(candidate) > max_words:
            break
        previous.pop()
        final.insert(0, candidate)

    if words_in_units(final) < min_words:
        chunks[-2] = previous + final
        chunks.pop()

    return chunks


def get_overlap(units: list[str], overlap_words: int) -> list[str]:
    selected: list[str] = []
    total = 0

    for unit in reversed(units):
        selected.append(unit)
        total += count_words(unit)
        if total >= overlap_words:
            break

    return list(reversed(selected))


def chunk_text(text: str) -> list[str]:
    units = split_into_units(text, CHUNK_MAX_WORDS)
    if not units:
        return []

    base_chunks = pack_chunks(units, CHUNK_MAX_WORDS)
    base_chunks = fix_short_final_chunk(
        base_chunks,
        MIN_FINAL_CHUNK_WORDS,
        CHUNK_MAX_WORDS,
    )

    chunks: list[str] = []

    for index, units_in_chunk in enumerate(base_chunks):
        prefix = (
            get_overlap(base_chunks[index - 1], CHUNK_OVERLAP_WORDS)
            if index > 0 and CHUNK_OVERLAP_WORDS > 0
            else []
        )
        chunk = normalize_content(" ".join(prefix + units_in_chunk))

        if chunk and (not chunks or chunk != chunks[-1]):
            chunks.append(chunk)

    return chunks

## Đọc input và chỉ giữ `{id, content, domain}`

In [ ]:
REQUIRED_FIELDS = {"id", "content", "domain"}
documents_by_split: dict[str, list[dict[str, str]]] = {}

for split, path in INPUT_FILES.items():
    raw_records = read_json_or_jsonl(path)
    clean_records: list[dict[str, str]] = []
    seen_ids: set[str] = set()

    for index, record in enumerate(raw_records):
        missing = REQUIRED_FIELDS - set(record)
        if missing:
            raise ValueError(
                f"{split}, record {index} thiếu: {sorted(missing)}"
            )

        doc_id = str(record["id"]).strip()
        domain = str(record["domain"]).strip()

        if not isinstance(record["content"], str):
            raise TypeError(f"{split}: content của {doc_id!r} không phải string.")

        content = normalize_content(record["content"])

        if not doc_id or not domain:
            raise ValueError(f"{split}: ID/domain rỗng tại record {index}.")

        if doc_id in seen_ids:
            raise ValueError(f"{split}: ID trùng: {doc_id}")

        seen_ids.add(doc_id)

        if content:
            clean_records.append({
                "id": doc_id,
                "content": content,
                "domain": domain,
            })

    documents_by_split[split] = clean_records
    print(f"{split:5s}: {len(clean_records):,} documents")


if CHECK_ID_LEAKAGE:
    ids = {
        split: {record["id"] for record in records}
        for split, records in documents_by_split.items()
    }

    split_names = list(ids)
    for left_index, left in enumerate(split_names):
        for right in split_names[left_index + 1:]:
            overlap = ids[left] & ids[right]
            if overlap:
                raise ValueError(
                    f"ID leakage giữa {left} và {right}: "
                    f"{sorted(overlap)[:10]}"
                )

    print("Không có ID leakage giữa train/dev/test.")

## Tạo `chunks_train/dev/test.jsonl`

In [ ]:
chunks_by_split: dict[str, list[dict[str, str]]] = {}

for split, documents in documents_by_split.items():
    output_records: list[dict[str, str]] = []
    seen_context_ids: set[str] = set()

    for document in tqdm(documents, desc=f"Chunking {split}"):
        chunks = chunk_text(document["content"])

        for chunk_index, chunk in enumerate(chunks):
            context_id = f"{document['id']}_chunk_{chunk_index:03d}"

            if context_id in seen_context_ids:
                raise ValueError(f"context_id trùng: {context_id}")

            seen_context_ids.add(context_id)

            output_records.append({
                "id": document["id"],
                "context_id": context_id,
                "content": chunk,
                "domain": document["domain"],
            })

    chunks_by_split[split] = output_records
    output_path = OUTPUT_DIR / f"chunks_{split}.jsonl"
    write_jsonl(output_records, output_path)

    print(f"{split:5s}: {len(output_records):,} chunks → {output_path}")

## Thống kê và kiểm tra schema

Thống kê nằm ở CSV riêng, không thêm metadata vào từng chunk.

In [ ]:
EXPECTED_FIELDS = {"id", "context_id", "content", "domain"}
all_context_ids: set[str] = set()
rows = []

for split, chunks in chunks_by_split.items():
    for chunk in chunks:
        assert set(chunk) == EXPECTED_FIELDS
        assert chunk["context_id"] not in all_context_ids
        all_context_ids.add(chunk["context_id"])

    domains = sorted({chunk["domain"] for chunk in chunks})

    for domain in domains:
        domain_chunks = [
            chunk for chunk in chunks
            if chunk["domain"] == domain
        ]
        word_counts = [
            count_words(chunk["content"])
            for chunk in domain_chunks
        ]
        document_count = len({
            chunk["id"] for chunk in domain_chunks
        })

        rows.append({
            "split": split,
            "domain": domain,
            "documents": document_count,
            "chunks": len(domain_chunks),
            "avg_chunks_per_document": (
                len(domain_chunks) / document_count
                if document_count else 0
            ),
            "min_words": min(word_counts) if word_counts else 0,
            "median_words": (
                float(pd.Series(word_counts).median())
                if word_counts else 0
            ),
            "max_words": max(word_counts) if word_counts else 0,
        })

statistics_df = pd.DataFrame(rows)
statistics_df.to_csv(
    OUTPUT_DIR / "chunk_statistics.csv",
    index=False,
    encoding="utf-8-sig",
)

display(statistics_df)
print("Schema hợp lệ. Tổng context_id:", f"{len(all_context_ids):,}")

for split in ("train", "dev", "test"):
    print(f"\n--- {split.upper()} SAMPLE ---")
    for sample in chunks_by_split[split][:2]:
        display(sample)

## Output dùng cho tạo ground truth

```text
04_chunks/
├── chunks_train.jsonl
├── chunks_dev.jsonl
├── chunks_test.jsonl
└── chunk_statistics.csv
```

Mỗi chunk có đúng bốn trường:

```json
{
  "id": "news_001",
  "context_id": "news_001_chunk_000",
  "content": "Nội dung chunk...",
  "domain": "news"
}
```

`task` chưa được thêm ở đây vì cùng một chunk có thể dùng cho cả QA và summarization. Khi tạo ground-truth sample, thêm:

```json
"task": "qa"
```

hoặc:

```json
"task": "sum"
```